# Notebook 2 — Dataset Finalisation
**When the Floor Speaks, Does the Market Listen?**
*Employee Sentiment, Stock Performance, and the Dual-Level Quality Signal*

Yulin Tian & Ilia Kornietskii | BI Norwegian Business School | GRA 19701 Master Thesis

---

**Purpose:** Construct the final thesis dataset by merging 86-firm employee reviews with `Industry Category` and `Stock Price (USD)`.

| Input | Detail |
|-------|--------|
| Reviews CSV | 1,898,445 rows × 22 columns (86 firms) |
| Industry mapping | `Targted Firms Industry Category.csv` |
| Stock prices | `stock_prices_2016_2023.csv` |
| **Output** | `Thesis_Dataset_Final.csv` (1,898,445 rows × 24 columns) |

In [ ]:
import pandas as pd
import calendar
from pathlib import Path

DATA_DIR     = Path(r'C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources')
REVIEWS_CSV  = DATA_DIR / '86firms_reviews_2016jan_2023mar.csv'
INDUSTRY_CSV = DATA_DIR / 'Targted Firms Industry Category.csv'
STOCK_CSV    = DATA_DIR / 'stock_prices_2016_2023.csv'
OUTPUT_CSV   = DATA_DIR / 'Thesis_Dataset_Final.csv'

# Name alignment: reference files use the pre-consolidation GM name;
NAME_ALIGN = {
    'General Motors GM': 'General Motors',
}

print('Paths configured.')


Paths configured.


In [2]:
df = pd.read_csv(REVIEWS_CSV, low_memory=False)
df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce')

print(f'Reviews loaded  : {len(df):,} rows  (expected 1,898,445)')
print(f'Unique firms    : {df["Company Name"].nunique()}  (expected 86)')
print(f'Date range      : {df["review_date"].min().date()}  →  {df["review_date"].max().date()}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
df.head(3)


Reviews loaded  : 1,898,445 rows  (expected 1,898,445)
Unique firms    : 86  (expected 86)
Date range      : 2016-01-01  →  2023-03-31
Columns (22): ['rating', 'title', 'status', 'pros', 'cons', 'advice', 'Recommend', 'CEO Approval', 'Business Outlook', 'Career Opportunities', 'Compensation and Benefits', 'Senior Management', 'Work/Life Balance', 'Culture & Values', 'Diversity & Inclusion', 'firm_link', 'date', 'job', 'index', 'Company Name', 'review_date', 'review_year']


,rating,title,status,pros,cons,advice,Recommend,CEO Approval,Business Outlook,Career Opportunities,...,Work/Life Balance,Culture & Values,Diversity & Inclusion,firm_link,date,job,index,Company Name,review_date,review_year
0,4.0,Great place to start your career,"Current Employee, more than 1 year",Growth mindset Intelligent colleagues Learning...,The pay is not up to industry standard for wor...,NaN,v,r,r,4.0,...,3.0,5.0,5.0,https://www.glassdoor.com/Reviews/Goldman-Sach...,"Jan 5, 2023",Senior Analyst,NaN,Goldman Sachs,2023-01-05,2023.0
1,5.0,Best job ever,"Former Employee, less than 1 year",10/10 best job I've ever had,I was a janitor so not the best job in the wor...,NaN,v,v,o,5.0,...,5.0,5.0,5.0,https://www.glassdoor.com/Reviews/Goldman-Sach...,"Mar 18, 2023",Janitor,NaN,Goldman Sachs,2023-03-18,2023.0
2,2.0,Just. Don't.,"Current Employee, more than 1 year","Great benefits, and 20-35 days of PTO dependin...",1) 5 days a week in-office (don't let them foo...,NaN,x,x,x,2.0,...,1.0,1.0,5.0,https://www.glassdoor.com/Reviews/Goldman-Sach...,"Jan 18, 2023",Associate Recruiter,NaN,Goldman Sachs,2023-01-18,2023.0


In [3]:
# ── Load only the two columns we need ────────────────────────────────────
ind_ref = pd.read_csv(INDUSTRY_CSV, usecols=['Company Name', 'Industry Category'])

# Align reference names → consolidated review names
ind_ref['Company Name'] = ind_ref['Company Name'].replace(NAME_ALIGN)

print(f'Industry reference: {len(ind_ref)} firms')
print('\nCategory distribution in reference:')
print(ind_ref['Industry Category'].value_counts().to_string())

# Left-join on Company Name
df = df.merge(ind_ref, on='Company Name', how='left')

print(f'\nRows after industry join: {len(df):,}  (must equal 1,898,445)')


Industry reference: 87 firms

Category distribution in reference:
Industry Category
Financial Services, Banking & Fintech         20
Technology (Hardware, Software & Internet)    14
E-Commerce & Retail                           13
IT Consulting & Professional Services         10
Telecommunications                             7
Automotive, Aerospace & Manufacturing          7
Fast Food & Restaurant Chains                  5
Healthcare, Pharmacy & Insurance               3
Consumer Goods & FMCG                          3
Logistics & Supply Chain                       2
Hospitality & Travel                           2
Energy & Resources                             1

Rows after industry join: 1,898,445  (must equal 1,898,445)


In [4]:
# ── Validation ───────────────────────────────────────────────────────────
null_ind = df['Industry Category'].isna().sum()
print(f'Null Industry Category values : {null_ind:,}')

if null_ind > 0:
    unmatched = df.loc[df['Industry Category'].isna(), 'Company Name'].unique()
    print('  Firms NOT matched in industry reference:')
    for nm in unmatched:
        print(f'    "{nm}"')
    print('  --> Add a NAME_ALIGN entry in Cell 1 to fix.')
else:
    print('  All 86 firms matched successfully.')

print(f'\nCategory distribution across {len(df):,} review rows:')
print(df['Industry Category'].value_counts().to_string())

# Spot-check: pick one firm and verify
spot = df.loc[df['Company Name'] == 'Amazon', 'Industry Category'].iloc[0]
print(f'\nSpot-check Amazon category: "{spot}"')


Null Industry Category values : 0
  All 86 firms matched successfully.

Category distribution across 1,898,445 review rows:
Industry Category
Technology (Hardware, Software & Internet)    466285
E-Commerce & Retail                           318974
IT Consulting & Professional Services         296701
Financial Services, Banking & Fintech         274979
Fast Food & Restaurant Chains                 158246
Automotive, Aerospace & Manufacturing         142124
Telecommunications                            104790
Healthcare, Pharmacy & Insurance               42372
Logistics & Supply Chain                       36761
Consumer Goods & FMCG                          26615
Hospitality & Travel                           22245
Energy & Resources                              8353

Spot-check Amazon category: "Technology (Hardware, Software & Internet)"


In [5]:
# ── Load wide-format stock prices ────────────────────────────────────────
sp_wide = pd.read_csv(STOCK_CSV)

# Align reference names → consolidated review names
sp_wide['Company Name'] = sp_wide['Company Name'].replace(NAME_ALIGN)

print(f'Stock price table: {len(sp_wide)} firms  |  {sp_wide.shape[1]} columns')

# Month columns are everything except Company Name and Ticker
month_cols = [c for c in sp_wide.columns if c not in ('Company Name', 'Ticker')]
print(f'Month columns   : {len(month_cols)}  ({month_cols[0]} … {month_cols[-1]})')

# Melt → long format: Company Name | month_label | Stock Price (USD)
sp_long = sp_wide.melt(
    id_vars=['Company Name'],
    value_vars=month_cols,
    var_name='month_label',
    value_name='Stock Price (USD)',
)
# Coerce 'N/A' strings and blanks to proper NaN, then float
sp_long['Stock Price (USD)'] = pd.to_numeric(sp_long['Stock Price (USD)'], errors='coerce')

print(f'Long table rows : {len(sp_long):,}  (expected {len(sp_wide)} firms × {len(month_cols)} months = {len(sp_wide)*len(month_cols):,})')
sp_long.head()


Stock price table: 86 firms  |  98 columns
Month columns   : 96  (2016 Jan … 2023 Dec)
Long table rows : 8,256  (expected 86 firms × 96 months = 8,256)


,Company Name,month_label,Stock Price (USD)
0,Amazon,2016 Jan,29.35
1,Walmart,2016 Jan,18.40
2,Cognizant Technology Solutions,2016 Jan,55.84
3,McDonald s,2016 Jan,96.49
4,Accenture,2016 Jan,90.11


In [6]:
# ── Build month_label for each review (e.g. '2016 Jan') ─────────────────
# review_date was already parsed in Cell 2
df['month_label'] = (
    df['review_date'].dt.year.astype('Int64').astype(str)
    + ' '
    + df['review_date'].dt.month.map(lambda m: calendar.month_abbr[int(m)])
)

print('Sample month_labels:', df['month_label'].head(5).tolist())

# ── Left-join stock prices on Company Name + month_label ─────────────────
df = df.merge(
    sp_long[['Company Name', 'month_label', 'Stock Price (USD)']],
    on=['Company Name', 'month_label'],
    how='left',
)

print(f'Rows after stock price join: {len(df):,}  (must equal 1,898,445)')


Sample month_labels: ['2023 Jan', '2023 Mar', '2023 Jan', '2023 Mar', '2023 Mar']
Rows after stock price join: 1,898,445  (must equal 1,898,445)


In [7]:
# ── Overall null count ───────────────────────────────────────────────────
null_sp = df['Stock Price (USD)'].isna().sum()
pct     = null_sp / len(df) * 100
print(f'Null Stock Price rows : {null_sp:,}  ({pct:.2f}% of total)')

# ── Null rows by firm — expected firms with partial/full N/A ──────────────
# WBA, VMW, JWN  → delisted; no data from Yahoo Finance (run patch_missing_tickers_tiingo.py to fix)
# Dell            → relisted Dec 28 2018; pre-IPO months set N/A
# Concentrix      → listed Dec 1 2020; Nov 2020 set N/A
# Uber            → IPO May 10 2019; pre-IPO months set N/A
# Victoria's Secret → spun off Aug 2 2021; pre-spinoff months set N/A
null_by_firm = (
    df[df['Stock Price (USD)'].isna()]
    .groupby('Company Name')
    .size()
    .sort_values(ascending=False)
    .rename('null_rows')
)
print('\nFirms with null stock prices (all should be on the expected list above):')
print(null_by_firm.to_string())

# ── Verify no unexpected firms appear ─────────────────────────────────────
EXPECTED_NULLS = {'Walgreens Boots Alliance', 'VMware', 'Nordstrom',
                  'Dell Technologies', 'Concentrix', 'Uber', "Victoria's Secret"}
unexpected = set(null_by_firm.index) - EXPECTED_NULLS
if unexpected:
    print(f'\nWARNING: unexpected firms with null prices: {unexpected}')
    print('  --> Check NAME_ALIGN mapping in Cell 1.')
else:
    print('\nAll null-price firms are on the expected list. Stock price join looks correct.')

# ── Spot-check: one known value ───────────────────────────────────────────
spot_price = df.loc[
    (df['Company Name'] == 'Amazon') & (df['month_label'] == '2023 Jan'),
    'Stock Price (USD)'
].mean()  # average within Jan-2023 (all rows same price)
print(f'\nSpot-check Amazon 2023 Jan stock price: {spot_price:.2f}  (expected ~103.13)')


Null Stock Price rows : 0  (0.00% of total)

Firms with null stock prices (all should be on the expected list above):
Series([], )

All null-price firms are on the expected list. Stock price join looks correct.

Spot-check Amazon 2023 Jan stock price: 103.13  (expected ~103.13)


In [8]:
# ── Drop month_label helper column (derived from review_date; not needed) ─
df = df.drop(columns=['month_label'])

# ── Final assertion ────────────────────────────────────────────────────────
assert len(df) == 1_898_445, f'Row count mismatch: got {len(df):,}'
assert 'Industry Category' in df.columns, 'Industry Category column missing'
assert 'Stock Price (USD)' in df.columns, 'Stock Price column missing'

print('=== Final Dataset Summary ===')
print(f'  Rows    : {len(df):,}')
print(f'  Columns : {len(df.columns)} → {list(df.columns)}')
print(f'  Firms   : {df["Company Name"].nunique()}')
print(f'  Industries: {df["Industry Category"].nunique()}')

print('\nFirst 5 rows (key columns):')
display(df[['Company Name', 'Industry Category', 'review_date', 'Stock Price (USD)']].head())

# ── Save ───────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\nSaved: {OUTPUT_CSV}')


=== Final Dataset Summary ===
  Rows    : 1,898,445
  Columns : 24 → ['rating', 'title', 'status', 'pros', 'cons', 'advice', 'Recommend', 'CEO Approval', 'Business Outlook', 'Career Opportunities', 'Compensation and Benefits', 'Senior Management', 'Work/Life Balance', 'Culture & Values', 'Diversity & Inclusion', 'firm_link', 'date', 'job', 'index', 'Company Name', 'review_date', 'review_year', 'Industry Category', 'Stock Price (USD)']
  Firms   : 86
  Industries: 12

First 5 rows (key columns):


,Company Name,Industry Category,review_date,Stock Price (USD)
0,Goldman Sachs,"Financial Services, Banking & Fintech",2023-01-05,337.49
1,Goldman Sachs,"Financial Services, Banking & Fintech",2023-03-18,303.94
2,Goldman Sachs,"Financial Services, Banking & Fintech",2023-01-18,337.49
3,Goldman Sachs,"Financial Services, Banking & Fintech",2023-03-30,303.94
4,Goldman Sachs,"Financial Services, Banking & Fintech",2023-03-23,303.94



Saved: C:\Users\tiany\OneDrive\GRA1971 Master Thesis\Data Resources\Thesis_Dataset_Final.csv
